# 02: Build the Master Table

This is the most important notebook in the project. Everything downstream reads
from the single table produced here, so getting the joins and feature engineering
right now means every later notebook is a one-liner.

Plan: start from `orders` (the fact table), left-join customers, items, payments,
and reviews onto it, then engineer the delivery-delay and freight-ratio features
that the rest of the analysis depends on.

In [5]:
import pandas as pd

orders = pd.read_csv("../data/interim/orders_clean.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
])
items_agg = pd.read_csv("../data/interim/items_agg.csv")
payments_agg = pd.read_csv("../data/interim/payments_agg.csv")
reviews = pd.read_csv("../data/interim/reviews_clean.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

print(orders.shape, items_agg.shape, payments_agg.shape, reviews.shape, customers.shape)

(99441, 8) (98666, 6) (99440, 4) (98673, 7) (99441, 5)


In [6]:
import os
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/interim", exist_ok=True)  # in case this one's also missing

## Join everything onto orders

All joins are `left` joins starting from `orders`, so we never lose an order even
if it has no items, no payment, or no review yet.

In [7]:
master = orders.merge(customers, on="customer_id", how="left")
master = master.merge(items_agg, on="order_id", how="left")
master = master.merge(payments_agg, on="order_id", how="left")
master = master.merge(
    reviews[["order_id", "review_score", "review_comment_message", "review_creation_date"]],
    on="order_id", how="left"
)

print("Master table shape:", master.shape)
assert master.shape[0] == orders.shape[0], "Row count changed — a join duplicated rows, investigate!"
master.head()

Master table shape: (99441, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_price,total_freight,n_sellers,n_products,total_payment_value,n_payment_methods,main_payment_type,review_score,review_comment_message,review_creation_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,1.0,1.0,38.71,2.0,voucher,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,1.0,1.0,141.46,1.0,boleto,4.0,Muito bom o produto.,2018-08-08
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,1.0,1.0,179.12,1.0,credit_card,5.0,NaN,2018-08-18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,1.0,1.0,72.20,1.0,credit_card,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,1.0,1.0,28.62,1.0,credit_card,5.0,NaN,2018-02-17


## Feature engineering

Four features drive almost the entire analysis:

- **`delivery_delay_days`**: delivered date minus *estimated* delivery date.
  Positive = late, negative = early. This is the single most predictive feature
  in the whole dataset.
- **`delivery_time_days`**: total time from purchase to delivery (not compared
  to the estimate — just raw speed).
- **`freight_ratio`**: freight cost as a fraction of item price. A flat freight
  value doesn't tell you much on its own; whether it's 5% or 80% of the order
  value does.
- **`order_value_bucket`**: price bucketed for easier group comparisons.

In [8]:
master["delivery_delay_days"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.days

master["delivery_time_days"] = (
    master["order_delivered_customer_date"] - master["order_purchase_timestamp"]
).dt.days

master["is_late"] = master["delivery_delay_days"] > 0

master["freight_ratio"] = master["total_freight"] / master["total_price"]

master["order_value_bucket"] = pd.cut(
    master["total_price"],
    bins=[0, 50, 100, 200, 500, float("inf")],
    labels=["<50", "50-100", "100-200", "200-500", "500+"]
)

master[["delivery_delay_days", "delivery_time_days", "freight_ratio"]].describe()

,delivery_delay_days,delivery_time_days,freight_ratio
count,96476.000000,96476.000000,98666.000000
mean,-11.876881,12.094086,0.308389
std,10.183854,9.551746,0.314762
min,-147.000000,0.000000,0.000000
25%,-17.000000,6.000000,0.131864
50%,-12.000000,10.000000,0.224374
75%,-7.000000,15.000000,0.380191
max,188.000000,209.000000,21.447059


## Split off the "delivered" subset

Delivery-time and satisfaction analysis only makes sense for orders that were
actually delivered (canceled/unavailable/processing orders have no real delivery
date, so their `delivery_delay_days` is `NaN` anyway — this just makes the
filter explicit and named, so nobody accidentally analyzes 3,000+ irrelevant
rows later).

In [9]:
delivered = master[master["order_status"] == "delivered"].copy()
print(f"master: {len(master):,} rows | delivered: {len(delivered):,} rows")

master.to_csv("../data/processed/master_orders.csv", index=False)
delivered.to_csv("../data/processed/delivered_orders.csv", index=False)

delivered[["review_score", "delivery_delay_days", "freight_ratio"]].describe()

master: 99,441 rows | delivered: 96,478 rows


,review_score,delivery_delay_days,freight_ratio
count,95832.000000,96470.000000,96478.000000
mean,4.156545,-11.875889,0.308370
std,1.284370,10.182105,0.311610
min,1.000000,-147.000000,0.000000
25%,4.000000,-17.000000,0.132016
50%,5.000000,-12.000000,0.224374
75%,5.000000,-7.000000,0.380531
max,5.000000,188.000000,21.447059


## Output

Two analysis-ready files now sit in `data/processed/`:

- `master_orders.csv` — every order, all statuses (order-volume / status analysis)
- `delivered_orders.csv` — delivered orders only (delivery-time / satisfaction analysis)

From notebook 03 onward, every notebook starts with
`pd.read_csv("../data/processed/delivered_orders.csv")` and jumps straight into
analysis; no more merging.